# Axe 5 — Needle-in-a-Haystack (NIAH)

**Objectif :** montrer que NSA (branche compression) retrouve une information à *n'importe quelle position*
dans le contexte, là où la fenêtre glissante échoue dès que la needle dépasse W tokens.

### Protocole
- **Haystack** : séquence de tokens aléatoires (bruit gaussien)
- **Needle** : un token à la position `needle_pos` avec un signal fort dans quelques dimensions
- **Query** : alignée avec le signal — dot-product élevé uniquement avec la needle
- On mesure si chaque mécanisme sélectionne le bon bloc

### Mécanismes
| Mécanisme | Recall attendu |
|---|---|
| **Sliding Window** (W=64) | 1.0 si `needle_pos ≥ T-W`, sinon **0.0** |
| **NSA Compression** (top-k blocs) | **1.0 partout** — indépendant de la position |

> **Note :** Utilise `naive_nsa_compression()` — pas de Triton, fonctionne sur CPU et GPU.

In [ ]:
import os, sys, subprocess

try:
    import google.colab
    IN_COLAB = True
    REPO_URL  = 'https://github.com/YentlCollin/native-sparse-attention.git'
    REPO_NAME = 'native-sparse-attention'

    if not os.path.exists(REPO_NAME):
        subprocess.run(['git', 'clone', '--recursive', REPO_URL], check=True)
    else:
        subprocess.run(['git', '-C', REPO_NAME, 'pull'], check=True)
        subprocess.run(['git', '-C', REPO_NAME, 'submodule', 'update', '--init', '--recursive'], check=True)

    if os.path.basename(os.getcwd()) != REPO_NAME:
        os.chdir(REPO_NAME)

    subprocess.run([sys.executable, '-m', 'pip', 'install', 'ninja', 'packaging', 'einops', '-q'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '-q', '--no-build-isolation'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'matplotlib', 'einops', '-q'], check=True)
    print('Setup Colab OK ✓')
except ImportError:
    IN_COLAB = False
    print('Local')

os.makedirs('notebooks/figures', exist_ok=True)
FIGURE_DIR = 'notebooks/figures'

In [ ]:
import math
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from native_sparse_attention.ops.naive import compression, naive_nsa_compression

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
else:
    print('CPU mode — le notebook fonctionne sans GPU (plus lent pour T=2048)')

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
B           = 1
H           = 1      # KV heads
HQ          = 16     # Query heads (GQA ratio = 16)
D           = 64     # Head dimension

BLOCK_SIZE  = 64
BLOCK_COUNTS = 4     # Top-k blocs sélectionnés
WINDOW_SIZE  = 64    # Taille de la fenêtre glissante

# Signal : spike dans SIGNAL_DIMS dimensions
# k_noise = 0.1 → signal très distinctif → recall NSA = 1.0 partout (c'est le message clé)
SIGNAL_STR  = 10.0
SIGNAL_DIMS = 8
K_NOISE     = 0.1    # bruit sur les keys du haystack

N_SEEDS     = 20
CTX_LENGTHS = [256, 512, 1024, 2048]
DTYPE       = torch.float32
scale       = D ** -0.5

print(f'BLOCK_SIZE={BLOCK_SIZE}, top-k={BLOCK_COUNTS}, W={WINDOW_SIZE}')
print(f'Signal: STR={SIGNAL_STR}, DIMS={SIGNAL_DIMS}, K_NOISE={K_NOISE}')
print()
print('Message attendu :')
print('  NSA compression → recall ≈ 1.0 quelle que soit la position')
print('  Sliding Window  → recall = 0 dès que needle_pos < T - W')

In [ ]:
# ── Fonctions NIAH ─────────────────────────────────────────────────────────

def make_needle_sequence(T, needle_pos, seed=0, k_noise=K_NOISE):
    torch.manual_seed(seed)
    k = torch.randn(B, T, H,  D, dtype=DTYPE, device=DEVICE) * k_noise
    v = torch.randn(B, T, H,  D, dtype=DTYPE, device=DEVICE)
    q = torch.randn(B, T, HQ, D, dtype=DTYPE, device=DEVICE) * 0.1

    # Needle : spike dans les premières SIGNAL_DIMS dimensions
    k[0, needle_pos, :, :SIGNAL_DIMS] = SIGNAL_STR

    # Query alignée avec le spike (dernière position)
    q[0, -1, :, :SIGNAL_DIMS] = SIGNAL_STR
    q[0, -1, :, SIGNAL_DIMS:] = 0.0
    return q, k, v


def sw_hit(needle_pos, T):
    """La fenêtre glissante voit-elle la needle ?"""
    return float(needle_pos >= (T - 1) - WINDOW_SIZE + 1)


def nsa_hit(q, k, v, needle_pos):
    """NSA compression sélectionne-t-elle le bloc de la needle ?"""
    T = q.shape[1]
    needle_block = needle_pos // BLOCK_SIZE
    g_cmp = torch.ones(B, T, HQ, dtype=DTYPE, device=DEVICE)
    block_indices, _ = naive_nsa_compression(
        q=q, k=k, v=v, g_cmp=g_cmp,
        block_counts=BLOCK_COUNTS,
        block_size=BLOCK_SIZE,
        scale=scale
    )
    selected = block_indices[0, -1, 0]  # blocs choisis pour query[-1], head 0
    return float((selected == needle_block).any().item())


print('Fonctions définies.')

In [ ]:
# ── Démonstration visuelle : un exemple concret ────────────────────────────
T_DEMO       = 512
NEEDLE_DEMO  = T_DEMO // 4   # 25% du contexte

q_d, k_d, v_d = make_needle_sequence(T_DEMO, NEEDLE_DEMO, seed=42)

# Scores d'attention dense (pour visualiser l'importance de chaque token)
G     = HQ // H
q_lst = q_d[0, -1].float()                               # [HQ, D]
k_exp = k_d[0].repeat_interleave(G, dim=1).float()       # [T, HQ, D]
scores = torch.einsum('hd,thd->th', q_lst * scale, k_exp)
attn   = scores.softmax(0).mean(-1).cpu().numpy()        # [T]

# Blocs sélectionnés par NSA compression
g_cmp = torch.ones(B, T_DEMO, HQ, dtype=DTYPE, device=DEVICE)
bidx, _ = naive_nsa_compression(q_d, k_d, v_d, g_cmp, BLOCK_COUNTS, BLOCK_SIZE, scale)
sel_blocks  = bidx[0, -1, 0].cpu().tolist()
needle_block = NEEDLE_DEMO // BLOCK_SIZE
num_blocks   = T_DEMO // BLOCK_SIZE

found = needle_block in sel_blocks
sw_ok = bool(sw_hit(NEEDLE_DEMO, T_DEMO))
print(f'needle_pos={NEEDLE_DEMO} (bloc {needle_block}/{num_blocks-1})')
print(f'NSA compression → bloc trouvé : {found}   |   Sliding Window → atteint : {sw_ok}')
print(f'Blocs sélectionnés : {sorted(b for b in sel_blocks if b >= 0)}')

# Figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# (a) Profil d'attention dense
ax1.plot(attn, color='steelblue', lw=0.8, alpha=0.8, label='Attention dense')
ax1.axvline(NEEDLE_DEMO, color='red', lw=2, ls='--', label=f'Needle (pos {NEEDLE_DEMO})')
ax1.axvspan(T_DEMO - WINDOW_SIZE, T_DEMO, alpha=0.12, color='green', label=f'Sliding window (W={WINDOW_SIZE})')
ax1.set_xlim(0, T_DEMO); ax1.set_xlabel('Token'); ax1.set_ylabel('Attention weight')
ax1.set_title('(a) Attention dense — la needle domine')
ax1.legend(fontsize=8)

# (b) Score moyen par bloc après compression
block_scores = [attn[b*BLOCK_SIZE:(b+1)*BLOCK_SIZE].sum() for b in range(num_blocks)]
colors = []
for b in range(num_blocks):
    if b == needle_block:
        colors.append('limegreen' if found else 'red')
    elif b in sel_blocks:
        colors.append('orange')
    else:
        colors.append('lightgray')
ax2.bar(range(num_blocks), block_scores, color=colors, edgecolor='white', lw=0.5)
ax2.set_xlabel('Bloc'); ax2.set_ylabel('Somme attention')
ax2.set_title('(b) Sélection de blocs par NSA compression')
legend_elems = [
    Patch(facecolor='limegreen', label='Needle sélectionnée ✓'),
    Patch(facecolor='red',       label='Needle manquée ✗'),
    Patch(facecolor='orange',    label='Autre bloc sélectionné'),
    Patch(facecolor='lightgray', label='Non sélectionné'),
]
ax2.legend(handles=legend_elems, fontsize=8)

plt.suptitle(f'T={T_DEMO}, needle à {NEEDLE_DEMO}/{T_DEMO} ({100*NEEDLE_DEMO//T_DEMO}% du contexte)', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/axe5_demo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Benchmark : recall vs position de la needle ────────────────────────────
# Positions testées : de 2% à 99% pour inclure la boundary SW
print('Benchmark NIAH en cours...')
print(f'  {len(CTX_LENGTHS)} longueurs × 20 positions × {N_SEEDS} seeds')
print()

results = {}

for T in CTX_LENGTHS:
    num_blocks_T = T // BLOCK_SIZE
    sw_boundary  = (T - WINDOW_SIZE) / T   # position fractionnaire de la boundary SW

    # Positions : 10 points hors SW (0-97%) + 10 points dans SW (97-99%)
    # On force des points des deux côtés de la boundary pour un graphique clair
    outside = np.linspace(0.02, sw_boundary - 0.01, 12)
    inside  = np.linspace(sw_boundary + 0.005, 0.99, 8)
    fracs   = np.concatenate([outside, inside])

    pos_pcts, sw_recalls, nsa_recalls = [], [], []

    for frac in fracs:
        needle_pos = max(0, min(int(frac * T), T - BLOCK_SIZE - 1))
        if needle_pos < 0:
            continue

        sw_r = sw_hit(needle_pos, T)   # déterministe

        hits = []
        for seed in range(N_SEEDS):
            q, k, v = make_needle_sequence(T, needle_pos, seed=seed)
            hits.append(nsa_hit(q, k, v, needle_pos))
        nsa_r = float(np.mean(hits))

        pos_pcts.append(needle_pos / T * 100)
        sw_recalls.append(sw_r)
        nsa_recalls.append(nsa_r)

    results[T] = {
        'pos':  np.array(pos_pcts),
        'sw':   np.array(sw_recalls),
        'nsa':  np.array(nsa_recalls),
        'sw_boundary_pct': sw_boundary * 100,
    }
    print(f'T={T:5d}: NSA recall = {np.mean(nsa_recalls):.3f} | '
          f'SW recall = {np.mean(sw_recalls):.3f} | '
          f'SW boundary @ {sw_boundary*100:.1f}%')

print('\nBenchmark terminé.')

In [ ]:
# ── Figure principale : Recall vs position de la needle ───────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharey=True)
axes = axes.flatten()

for idx, T in enumerate(CTX_LENGTHS):
    ax  = axes[idx]
    r   = results[T]
    bnd = r['sw_boundary_pct']

    # Zone sliding window
    ax.axvspan(bnd, 100, alpha=0.10, color='green')
    ax.axvline(bnd, color='green', lw=1.5, ls='--', alpha=0.8,
               label=f'SW boundary ({bnd:.0f}%)')

    # NSA compression recall
    ax.plot(r['pos'], r['nsa'], 'o-', color='darkorange', lw=2, ms=5,
            label=f'NSA compression (top-{BLOCK_COUNTS})')

    # Sliding Window recall
    ax.plot(r['pos'], r['sw'], 's--', color='green', lw=1.5, ms=4,
            label=f'Sliding Window (W={WINDOW_SIZE})')

    ax.set_xlim(0, 100)
    ax.set_ylim(-0.05, 1.15)
    ax.set_xlabel('Position needle (% du contexte)')
    ax.set_ylabel('Recall')
    ax.set_title(f'T = {T} tokens ({T//BLOCK_SIZE} blocs)')
    ax.legend(fontsize=7.5)
    ax.grid(True, alpha=0.3)
    ax.axhline(1.0, color='gray', lw=0.8, ls=':', alpha=0.6)

    # Annotation
    ax.annotate('SW: recall=0\n(needle hors fenêtre)',
                xy=(bnd/2, 0.05), fontsize=7, color='green', ha='center')
    ax.annotate('NSA: recall=1\n(position-agnostique)',
                xy=(bnd/2, 0.85), fontsize=7, color='darkorange', ha='center')

plt.suptitle(
    'NIAH : NSA Compression vs Sliding Window\n'
    f'BLOCK_SIZE={BLOCK_SIZE}, top-k={BLOCK_COUNTS}, W={WINDOW_SIZE}',
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/axe5_recall_vs_position.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegardé.')

In [ ]:
# ── Courbe : recall NSA selon la force du signal (régime réaliste) ─────────
# Ici on augmente le bruit (k_noise=1.0) pour montrer que la mean pooling
# peut diluer un signal faible → recall < 1.0
# Cela justifie l'entraînement des gates g_cmp dans le vrai modèle NSA.

T_SIG        = 1024
NEEDLE_SIG   = T_SIG // 4    # 25% — hors SW
K_NOISE_HIGH = 1.0           # bruit réaliste (standard Gaussian)
STRENGTHS    = [0.5, 1.0, 2.0, 3.0, 5.0, 8.0, 10.0, 15.0, 20.0]
N_SEEDS_SIG  = 40

print(f'T={T_SIG}, needle@{100*NEEDLE_SIG//T_SIG}%, k_noise={K_NOISE_HIGH}')
print(f'SW reach : {bool(sw_hit(NEEDLE_SIG, T_SIG))} (attendu False)')
print()

recalls_sig = []
for strength in STRENGTHS:
    hits = []
    for seed in range(N_SEEDS_SIG):
        q, k, v = make_needle_sequence(T_SIG, NEEDLE_SIG, seed=seed, k_noise=K_NOISE_HIGH)
        # Override signal strength
        k[0, NEEDLE_SIG, :, :SIGNAL_DIMS] = strength
        q[0, -1, :, :SIGNAL_DIMS] = strength
        q[0, -1, :, SIGNAL_DIMS:] = 0.0
        hits.append(nsa_hit(q, k, v, NEEDLE_SIG))
    r = float(np.mean(hits))
    recalls_sig.append(r)
    print(f'  signal={strength:5.1f} → recall = {r:.3f}')

# Baseline aléatoire
random_baseline = BLOCK_COUNTS / (T_SIG // BLOCK_SIZE)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogx(STRENGTHS, recalls_sig, 'D-', color='darkorange', lw=2, ms=8,
            label='NSA compression recall')
ax.axhline(1.0, color='green', lw=1, ls='--', label='Recall parfait')
ax.axhline(random_baseline, color='gray', lw=1, ls=':', 
           label=f'Baseline aléatoire ({BLOCK_COUNTS}/{T_SIG//BLOCK_SIZE} = {random_baseline:.2f})')
ax.axvline(SIGNAL_STR, color='steelblue', lw=1, ls='--', alpha=0.7,
           label=f'Signal utilisé dans exp principale ({SIGNAL_STR})')

ax.set_xlabel('Force du signal needle (log scale)')
ax.set_ylabel('Recall')
ax.set_title(
    f'Recall NSA compression vs force du signal\n'
    f'(T={T_SIG}, needle@25%, bruit={K_NOISE_HIGH}, top-{BLOCK_COUNTS}/{T_SIG//BLOCK_SIZE} blocs)'
)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.1)

# Zone critique
ax.fill_between([STRENGTHS[0], STRENGTHS[-1]], 0, random_baseline,
                alpha=0.05, color='red', label='_')
ax.text(0.6, random_baseline + 0.02, 'Signal < bruit', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/axe5_recall_vs_signal.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegardé.')

In [ ]:
# ── Heatmap : NSA vs SW — toutes positions × toutes longueurs ─────────────
N_POS_HEAT = 15
fracs_heat = np.linspace(0.02, 0.99, N_POS_HEAT)

nsa_mat = np.zeros((N_POS_HEAT, len(CTX_LENGTHS)))
sw_mat  = np.zeros((N_POS_HEAT, len(CTX_LENGTHS)))

for j, T in enumerate(CTX_LENGTHS):
    for i, frac in enumerate(fracs_heat):
        needle_pos = max(0, min(int(frac * T), T - BLOCK_SIZE - 1))
        sw_mat[i, j] = sw_hit(needle_pos, T)
        # Utilise les résultats déjà calculés si disponibles, sinon recalcule
        r = results[T]
        closest = np.argmin(np.abs(r['pos'] - frac * 100))
        nsa_mat[i, j] = r['nsa'][closest]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, mat, title, cmap in [
    (axes[0], nsa_mat, 'NSA Compression (recall)', 'YlOrRd'),
    (axes[1], sw_mat,  'Sliding Window (recall)',  'YlGn'),
]:
    im = ax.imshow(mat, origin='lower', aspect='auto', vmin=0, vmax=1,
                   cmap=cmap, interpolation='nearest')
    ax.set_xticks(range(len(CTX_LENGTHS)))
    ax.set_xticklabels([str(T) for T in CTX_LENGTHS])
    ax.set_yticks(range(N_POS_HEAT))
    ax.set_yticklabels([f'{100*f:.0f}%' for f in fracs_heat])
    ax.set_xlabel('Longueur de contexte T')
    ax.set_ylabel('Position needle')
    ax.set_title(title)
    plt.colorbar(im, ax=ax)
    for i in range(N_POS_HEAT):
        for j in range(len(CTX_LENGTHS)):
            v = mat[i, j]
            ax.text(j, i, f'{v:.1f}', ha='center', va='center',
                    fontsize=7, color='white' if v > 0.5 else 'black')

plt.suptitle('Heatmap NIAH : NSA compression vs Sliding Window', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/axe5_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegardé.')

In [ ]:
# ── Synthèse ───────────────────────────────────────────────────────────────
print('=' * 58)
print('RÉSULTATS NEEDLE-IN-A-HAYSTACK')
print('=' * 58)
print(f'BLOCK_SIZE={BLOCK_SIZE}, top-k={BLOCK_COUNTS}, W={WINDOW_SIZE}\n')

for T in CTX_LENGTHS:
    r   = results[T]
    bnd = r['sw_boundary_pct'] / 100
    out = r['pos'] / 100 < bnd
    ins = ~out
    print(f'T={T:5d} | NSA hors_SW={r["nsa"][out].mean():.3f} '
          f'dans_SW={r["nsa"][ins].mean():.3f} | '
          f'SW hors_SW={r["sw"][out].mean():.3f} '
          f'dans_SW={r["sw"][ins].mean():.3f}')

print()
print('Conclusion :')
print('  - SW recall = 0.0 hors fenêtre (quel que soit T)')
print('  - NSA recall ≈ 1.0 partout — accès position-agnostique')
print('  - La branche compression résout structurellement NIAH')
print('  - Un signal faible (< bruit) peut diluer la mean pooling')
print('    → justifie l\'entraînement des gates g_cmp dans NSA réel')